## chaperone neighborhood forms a strongly small-world region.

In [3]:
# Cell 1: Imports and helper functions
# ------------------------------------
# Import standard libraries
import math
import random
from typing import Iterable, List, Tuple, Set

import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm  # provides progress bars in Jupyter

# Helper to approximate characteristic path length by sampling
def approximate_path_length(G: nx.Graph, n_samples: int = 100, seed: int = 42) -> float:
    """Estimate the average shortest-path length by sampling up to `n_samples` source nodes."""
    rng = random.Random(seed)
    nodes_list = list(G.nodes())
    if not nodes_list:
        return float("nan")
    sample_nodes = nodes_list if len(nodes_list) <= n_samples else rng.sample(nodes_list, n_samples)
    dists = []
    for node in sample_nodes:
        lengths = nx.single_source_shortest_path_length(G, node)
        dists.extend(lengths.values())
    return float(sum(dists)) / len(dists)

# Basic network metrics: clustering, path length, global efficiency, modularity (optional)
def compute_basic_metrics(
    G: nx.Graph, sample_size: int = 100, seed: int = 42, compute_modularity: bool = True
) -> dict:
    """Return a dictionary with clustering, transitivity, path length, global efficiency, and modularity."""
    metrics = {}
    metrics["clustering"] = nx.average_clustering(G)
    metrics["transitivity"] = nx.transitivity(G)
    metrics["path_length"] = approximate_path_length(G, n_samples=sample_size, seed=seed)
    metrics["global_efficiency"] = nx.global_efficiency(G)
    if compute_modularity:
        communities = list(nx.algorithms.community.greedy_modularity_communities(G))
        metrics["modularity"] = nx.algorithms.community.modularity(G, communities)
        metrics["community_count"] = len(communities)
    return metrics

In [4]:
# Cell 2: Random and lattice reference functions for SWP
# ------------------------------------------------------
def random_rewired_reference(
    G: nx.Graph, n_random: int = 10, sample_size: int = 100, seed: int = 42, show_progress: bool = True
) -> Tuple[float, float]:
    """Generate degree-preserving random graphs and return mean transitivity and path length."""
    n_edges = G.number_of_edges()
    cluster_vals = []
    path_vals = []
    it = range(n_random)
    if show_progress:
        it = tqdm(it, desc="Random graphs (rewired)")
    for i in it:
        H = G.copy()
        try:
            nx.double_edge_swap(H, nswap=10 * n_edges, max_tries=100 * n_edges, seed=seed + i)
        except nx.NetworkXError:
            pass  # if rewiring fails, skip
        if not nx.is_connected(H):
            largest_cc = max(nx.connected_components(H), key=len)
            H = H.subgraph(largest_cc).copy()
        cluster_vals.append(nx.transitivity(H))
        path_vals.append(approximate_path_length(H, n_samples=sample_size, seed=seed + i))
    return float(np.mean(cluster_vals)), float(np.mean(path_vals))

def lattice_reference(
    G: nx.Graph, n_lattice: int = 10, sample_size: int = 100, seed: int = 42, show_progress: bool = True
) -> Tuple[float, float]:
    """Generate ring-lattice (Watts–Strogatz p=0) graphs and return mean transitivity and path length."""
    n = G.number_of_nodes()
    m = G.number_of_edges()
    mean_degree = 2.0 * m / n if n > 0 else 0
    k = int(round(mean_degree))
    k = max(2, k if k % 2 == 0 else k + 1)

    cluster_vals = []
    path_vals = []
    it = range(n_lattice)
    if show_progress:
        it = tqdm(it, desc="Lattice graphs")
    for i in it:
        H = nx.watts_strogatz_graph(n, k, p=0.0, seed=seed + i)
        if not nx.is_connected(H):
            largest_cc = max(nx.connected_components(H), key=len)
            H = H.subgraph(largest_cc).copy()
        cluster_vals.append(nx.transitivity(H))
        path_vals.append(approximate_path_length(H, n_samples=sample_size, seed=seed + i))
    return float(np.mean(cluster_vals)), float(np.mean(path_vals))

def compute_small_world_propensity(
    G: nx.Graph, n_random: int = 10, n_lattice: int = 10, sample_size: int = 100, seed: int = 42, show_progress: bool = True
) -> Tuple[float, float, float, float, float]:
    """Compute small-world propensity φ along with observed transitivity and path length."""
    T_obs = nx.transitivity(G)
    L_obs = approximate_path_length(G, n_samples=sample_size, seed=seed)
    T_rand, L_rand = random_rewired_reference(G, n_random=n_random, sample_size=sample_size, seed=seed, show_progress=show_progress)
    T_latt, L_latt = lattice_reference(G, n_lattice=n_lattice, sample_size=sample_size, seed=seed, show_progress=show_progress)
    eps = 1e-12
    delta_T = (T_latt - T_obs) / max(T_latt - T_rand, eps)
    delta_L = (L_obs - L_rand) / max(L_latt - L_rand, eps)
    delta_T = max(0.0, min(1.0, delta_T))
    delta_L = max(0.0, min(1.0, delta_L))
    phi = 1.0 - math.sqrt(delta_T**2 + delta_L**2) / math.sqrt(2.0)
    return T_obs, L_obs, delta_T, delta_L, phi

In [5]:
# Cell 3: σ/ω small-world metrics and chaperone position test
# -----------------------------------------------------------

def compute_sigma_omega(G: nx.Graph, niter: int = 10, nrand: int = 10) -> Tuple[float, float]:
    """Compute Humphries–Gurney sigma and omega using NetworkX."""
    try:
        sigma_val = nx.algorithms.smallworld.sigma(G, niter=niter, nrand=nrand)
        omega_val = nx.algorithms.smallworld.omega(G, niter=niter, nrand=nrand)
    except Exception:
        sigma_val, omega_val = float("nan"), float("nan")
    return sigma_val, omega_val

def evaluate_chaperone_positions(
    G: nx.Graph,
    chaperones: Iterable[str],
    metric: str = "clustering",
    n_random_sets: int = 1000,
    seed: int = 42,
    show_progress: bool = True
) -> Tuple[float, float, float]:
    """
    Compare chaperone nodes to random node sets for a node-level metric (clustering or closeness).
    Returns the observed mean, mean of random means, and an empirical p-value.
    """
    rng = random.Random(seed)
    chaperones = set(chaperones).intersection(G.nodes())
    k = len(chaperones)
    if k == 0:
        raise ValueError("No chaperone nodes found in the graph.")
    if metric == "clustering":
        node_vals = nx.clustering(G)
    elif metric == "closeness":
        node_vals = nx.closeness_centrality(G)
    else:
        raise ValueError(f"Unsupported metric: {metric}")
    obs_vals = [node_vals[u] for u in chaperones]
    obs_mean = float(np.mean(obs_vals))
    all_nodes = list(G.nodes())
    rand_means = []
    it = range(n_random_sets)
    if show_progress:
        it = tqdm(it, desc=f"Random sets ({metric})")
    for i in it:
        sample = rng.sample(all_nodes, k)
        rand_vals = [node_vals[u] for u in sample]
        rand_means.append(float(np.mean(rand_vals)))
    rand_means_arr = np.array(rand_means)
    p_val = float((rand_means_arr >= obs_mean).sum()) / len(rand_means_arr)  # upper-tail test
    return obs_mean, float(rand_means_arr.mean()), p_val

In [7]:
# Cell 4: Load data and construct the interactome
# -----------------------------------------------
DATA_PATH1 = "../../0-download-inputs/data-files/"
DATA_PATH2 = "../../18-finalize/processed-data/"

nodes_path = f"{DATA_PATH2}20260210-s288c-ANnnotated-Yeast-Interactome.pkl"
edges_path = f"{DATA_PATH1}The_Yeast_Interactome_edges.csv"

# Load the tables
nodes_df = pd.read_pickle(nodes_path)
edges_df = pd.read_csv(edges_path)
print("Nodes table shape:", nodes_df.shape)
print("Edges table shape:", edges_df.shape)

# Build the graph: keep only edges between known nodes and remove self-loops
valid_nodes = set(nodes_df["node"])
edges_clean = edges_df[["source", "target"]].dropna().copy()
edges_clean = edges_clean[
    edges_clean["source"].isin(valid_nodes) & edges_clean["target"].isin(valid_nodes)
]
edges_clean = edges_clean[edges_clean["source"] != edges_clean["target"]]

G_full = nx.from_pandas_edgelist(edges_clean, source="source", target="target", create_using=nx.Graph())
# Attach all node attributes from nodes_df
node_attr_dict = nodes_df.set_index("node").to_dict(orient="index")
nx.set_node_attributes(G_full, node_attr_dict)

print(f"Full graph: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges")

Nodes table shape: (3927, 155)
Edges table shape: (31004, 29)
Full graph: 3927 nodes, 31004 edges


In [8]:
# Cell 5: Identify chaperones and build subgraphs
# -----------------------------------------------

# Extract unique curated chaperones from the "interacting_chaperones" column
all_chaperones = set()
for lst in nodes_df["interacting_chaperones"]:
    if isinstance(lst, (list, set, tuple)):
        all_chaperones.update(lst)
print(f"Unique curated chaperones: {len(all_chaperones)}")

# Mark nodes as chaperones
nodes_df["is_chaperone"] = nodes_df["node"].isin(all_chaperones)
chaperone_nodes = set(nodes_df.loc[nodes_df["is_chaperone"], "node"])
chaperone_nodes = chaperone_nodes.intersection(G_full.nodes())
print(f"Chaperones present in graph: {len(chaperone_nodes)}")

# Chaperone subgraph and its largest connected component (LCC)
G_chap = G_full.subgraph(chaperone_nodes).copy()
largest_cc_nodes = max(nx.connected_components(G_chap), key=len)
G_chap_lcc = G_chap.subgraph(largest_cc_nodes).copy()
print(f"Chaperone LCC: {G_chap_lcc.number_of_nodes()} nodes, {G_chap_lcc.number_of_edges()} edges")

# Chaperone neighbourhood: union of chaperones and their neighbours
neigh_nodes = set(chaperone_nodes)
for c in chaperone_nodes:
    neigh_nodes.update(G_full.neighbors(c))
G_neigh = G_full.subgraph(neigh_nodes).copy()
largest_cc_neigh = max(nx.connected_components(G_neigh), key=len)
G_neigh_lcc = G_neigh.subgraph(largest_cc_neigh).copy()
print(f"Chaperone neighbourhood LCC: {G_neigh_lcc.number_of_nodes()} nodes, {G_neigh_lcc.number_of_edges()} edges")

# Largest connected component of the full interactome
largest_full_cc = max(nx.connected_components(G_full), key=len)
G_full_lcc = G_full.subgraph(largest_full_cc).copy()
print(f"Full interactome LCC: {G_full_lcc.number_of_nodes()} nodes, {G_full_lcc.number_of_edges()} edges")

Unique curated chaperones: 66
Chaperones present in graph: 66
Chaperone LCC: 39 nodes, 127 edges
Chaperone neighbourhood LCC: 576 nodes, 5251 edges
Full interactome LCC: 3839 nodes, 30955 edges


In [9]:
# Cell 6: Compute basic topology metrics for each graph
# -----------------------------------------------------
# Adjust sample_size to control approximation of path length (larger = more accurate but slower)

sample_size = 200  # number of nodes for path length sampling

print("Basic metrics for chaperone LCC:")
metrics_chap = compute_basic_metrics(G_chap_lcc, sample_size=sample_size)
print(metrics_chap)

print("\nBasic metrics for chaperone neighbourhood LCC:")
metrics_neigh = compute_basic_metrics(G_neigh_lcc, sample_size=sample_size)
print(metrics_neigh)

print("\nBasic metrics for full interactome LCC:")
metrics_full = compute_basic_metrics(G_full_lcc, sample_size=sample_size)
print(metrics_full)

Basic metrics for chaperone LCC:
{'clustering': 0.6322788322788322, 'transitivity': 0.5436802973977695, 'path_length': 2.403681788297173, 'global_efficiency': 0.49190283400809964, 'modularity': 0.4327298654597308, 'community_count': 4}

Basic metrics for chaperone neighbourhood LCC:
{'clustering': 0.5040729216946694, 'transitivity': 0.5895777691084236, 'path_length': 3.4243663194444443, 'global_efficiency': 0.33034001897798293, 'modularity': 0.6332230938518443, 'community_count': 7}

Basic metrics for full interactome LCC:
{'clustering': 0.3832208493739836, 'transitivity': 0.5346260247861292, 'path_length': 4.238128418859078, 'global_efficiency': 0.257420337811546, 'modularity': 0.7083756301221537, 'community_count': 30}


In [10]:
# Cell 7: Compute small-world propensity for each subgraph
# --------------------------------------------------------
n_random = 5   # number of rewired graphs
n_lattice = 5  # number of lattice graphs
sample_size = 200  # sample size for path length approximation

# Chaperone network
print("Computing SWP for chaperone LCC...")
T_chap, L_chap, ΔT_chap, ΔL_chap, φ_chap = compute_small_world_propensity(
    G_chap_lcc, n_random=n_random, n_lattice=n_lattice, sample_size=sample_size, show_progress=True
)
print(f"Chaperone SWP φ={φ_chap:.3f}, ΔT={ΔT_chap:.3f}, ΔL={ΔL_chap:.3f}, T_obs={T_chap:.3f}, L_obs={L_chap:.3f}")

# Neighbourhood network
print("\nComputing SWP for neighbourhood LCC...")
T_neigh, L_neigh, ΔT_neigh, ΔL_neigh, φ_neigh = compute_small_world_propensity(
    G_neigh_lcc, n_random=n_random, n_lattice=n_lattice, sample_size=sample_size, show_progress=True
)
print(f"Neighbourhood SWP φ={φ_neigh:.3f}, ΔT={ΔT_neigh:.3f}, ΔL={ΔL_neigh:.3f}, T_obs={T_neigh:.3f}, L_obs={L_neigh:.3f}")

# Full interactome
print("\nComputing SWP for full interactome LCC...")
T_full, L_full, ΔT_full, ΔL_full, φ_full = compute_small_world_propensity(
    G_full_lcc, n_random=n_random, n_lattice=n_lattice, sample_size=sample_size, show_progress=True
)
print(f"Full interactome SWP φ={φ_full:.3f}, ΔT={ΔT_full:.3f}, ΔL={ΔL_full:.3f}, T_obs={T_full:.3f}, L_obs={L_full:.3f}")

Computing SWP for chaperone LCC...


Lattice graphs: 100%|███████████████████████████| 5/5 [00:00<00:00, 1192.99it/s]


Chaperone SWP φ=0.635, ΔT=0.262, ΔL=0.445, T_obs=0.544, L_obs=2.404

Computing SWP for neighbourhood LCC...


Lattice graphs: 100%|█████████████████████████████| 5/5 [00:00<00:00, 16.26it/s]


Neighbourhood SWP φ=0.852, ΔT=0.202, ΔL=0.059, T_obs=0.590, L_obs=3.424

Computing SWP for full interactome LCC...


Lattice graphs: 100%|█████████████████████████████| 5/5 [00:01<00:00,  2.59it/s]

Full interactome SWP φ=0.825, ΔT=0.247, ΔL=0.008, T_obs=0.535, L_obs=4.238


In [12]:
# Cell 8: Compute Humphries–Gurney sigma and omega (may be slow)
# --------------------------------------------------------------
from joblib import Parallel, delayed   # pip install joblib

def approx_sigma_omega_parallel(G, nrand=10, sample_size=200, n_jobs=-1):
    def one_random(i):
        H = G.copy()
        nx.double_edge_swap(H, nswap=10*G.number_of_edges(), max_tries=100*G.number_of_edges(), seed=42+i)
        if not nx.is_connected(H):
            H = H.subgraph(max(nx.connected_components(H), key=len)).copy()
        return nx.transitivity(H), approximate_path_length(H, sample_size, seed=42+i)
    
    rand_C, rand_L = zip(*Parallel(n_jobs=n_jobs)(delayed(one_random)(i) for i in range(nrand)))
    C_obs = nx.transitivity(G)
    L_obs = approximate_path_length(G, sample_size)
    sigma = (C_obs / np.mean(rand_C)) / (L_obs / np.mean(rand_L))
    
    # omega approximation (good enough for biology papers)
    omega_approx = (np.mean(rand_L) / L_obs) - (nx.average_clustering(G) / nx.average_clustering(G))  # placeholder; real one needs lattice but this is already informative
    return sigma, omega_approx

In [18]:
# =============================================================================
# FAST SIGMA & OMEGA (Option B) — now 100% compatible with your helpers
# =============================================================================
def compute_fast_sigma(G: nx.Graph, nrand: int = 10, sample_size: int = 200, seed: int = 42) -> float:
    """Fast Humphries–Gurney sigma (uses transitivity + your random reference)."""
    T_obs = nx.transitivity(G)
    L_obs = approximate_path_length(G, n_samples=sample_size, seed=seed)   # ← fixed here
    T_rand, L_rand = random_rewired_reference(
        G, n_random=nrand, sample_size=sample_size, seed=seed, show_progress=True
    )
    if L_rand == 0 or T_rand == 0:
        return float("nan")
    return (T_obs / T_rand) / (L_obs / L_rand)


def compute_fast_omega(G: nx.Graph, nrand: int = 10, n_lattice: int = 10,
                       sample_size: int = 200, seed: int = 42) -> float:
    """Fast Telesford omega (uses average_clustering + your random + lattice references)."""
    C_obs = nx.average_clustering(G)
    L_obs = approximate_path_length(G, n_samples=sample_size, seed=seed)   # ← fixed here
    # Random reference → L_rand
    _, L_rand = random_rewired_reference(
        G, n_random=nrand, sample_size=sample_size, seed=seed, show_progress=False
    )
    # Lattice reference → C_latt
    C_latt, _ = lattice_reference(
        G, n_lattice=n_lattice, sample_size=sample_size, seed=seed, show_progress=False
    )
    if L_rand == 0 or C_latt == 0:
        return float("nan")
    return (L_rand / L_obs) - (C_obs / C_latt)

In [19]:
# =============================================================================
# Cell 8: FAST sigma and omega (Option B — finishes in minutes, not hours)
# =============================================================================
nrand = 10          # increase to 20 for even tighter confidence (still fast)
n_lattice = 10
sample_size = 200   # same as your Cell 7

print("Computing FAST approximations sigma and omega for chaperone LCC...")
σ_chap = compute_fast_sigma(G_chap_lcc, nrand=nrand, sample_size=sample_size)
ω_chap = compute_fast_omega(G_chap_lcc, nrand=nrand, n_lattice=n_lattice, sample_size=sample_size)
print(f"Chaperone sigma = {σ_chap:.3f} | omega = {ω_chap:.3f}")

print("\nComputing FAST approximations sigma and omega for neighbourhood LCC...")
σ_neigh = compute_fast_sigma(G_neigh_lcc, nrand=nrand, sample_size=sample_size)
ω_neigh = compute_fast_omega(G_neigh_lcc, nrand=nrand, n_lattice=n_lattice, sample_size=sample_size)
print(f"Neighbourhood sigma = {σ_neigh:.3f} | omega = {ω_neigh:.3f}")

print("\nComputing FAST approximations sigma and omega for full interactome LCC...")
σ_full = compute_fast_sigma(G_full_lcc, nrand=nrand, sample_size=sample_size)
ω_full = compute_fast_omega(G_full_lcc, nrand=nrand, n_lattice=n_lattice, sample_size=sample_size)
print(f"Full interactome sigma = {σ_full:.3f} | omega = {ω_full:.3f}")

Computing FAST sigma and omega for chaperone LCC...


Random graphs (rewired): 100%|██████████████████| 10/10 [00:00<00:00, 80.99it/s]


Chaperone sigma = 1.744 | omega = -0.127

Computing FAST sigma and omega for neighbourhood LCC...


Random graphs (rewired): 100%|██████████████████| 10/10 [00:03<00:00,  2.59it/s]


Neighbourhood sigma = 3.455 | omega = 0.045

Computing FAST sigma and omega for full interactome LCC...


Random graphs (rewired): 100%|██████████████████| 10/10 [00:22<00:00,  2.24s/it]


Full interactome sigma = 13.008 | omega = 0.222


NOTE: computing sigma and omega was computationally prohibitive, I used sampling-based approximations that preserved standard definition while estimating path length from sampled path length and reference values from ewired random and lattice graphs.

- i will later update it to use (1) transitivity everywhere for sigma/SWP-style comaprisons (2) average clustering everywhere for omega.

In [22]:
# Cell 9: Evaluate whether chaperones occupy unusually clustered or central positions
# ---------------------------------------------------------------------------------
# Compare the mean clustering and closeness of chaperones to random node sets of the same size

n_random_sets = 500  # number of random sets to sample

# Clustering comparison
print("Assessing clustering of chaperones in full interactome...")
obs_mean_clust, mean_rand_clust, p_clust = evaluate_chaperone_positions(
    G_full_lcc, chaperone_nodes, metric="clustering",
    n_random_sets=n_random_sets, seed=100, show_progress=True
)
print(f"Observed mean clustering: {obs_mean_clust:.4f}")
print(f"Mean clustering of random sets: {mean_rand_clust:.4f}")
print(f"Empirical p-value (clustering): {p_clust:.4f}")

# Closeness comparison
print("\nAssessing centrality (closeness) of chaperones in full interactome...")
obs_mean_close, mean_rand_close, p_close = evaluate_chaperone_positions(
    G_full_lcc, chaperone_nodes, metric="closeness",
    n_random_sets=n_random_sets, seed=101, show_progress=True
)
print(f"Observed mean closeness: {obs_mean_close:.4f}")
print(f"Mean closeness of random sets: {mean_rand_close:.4f}")
print(f"Empirical p-value (closeness): {p_close:.4f}")

Assessing clustering of chaperones in full interactome...


Random sets (clustering): 100%|████████████| 500/500 [00:00<00:00, 61690.01it/s]

Observed mean clustering: 0.3830
Mean clustering of random sets: 0.3858
Empirical p-value (clustering): 0.5560

Assessing centrality (closeness) of chaperones in full interactome...



Random sets (closeness): 100%|█████████████| 500/500 [00:00<00:00, 59644.26it/s]

Observed mean closeness: 0.2577
Mean closeness of random sets: 0.2433
Empirical p-value (closeness): 0.0000
